In [ ]:
import torchvision
import os
import numpy as np
import torch
from torch.utils.data import Subset

import torchvision.transforms as transforms
import torchvision.models as models

from torch.utils.data import DataLoader, Subset, ConcatDataset
from sklearn.metrics import classification_report, accuracy_score
from PIL import Image
from tqdm import tqdm

import json

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
PROJECT_ROOT = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "ProfessionAI_AIengineering/9. Generative AI/"
    "Project_Generative_AI"
)

# Load train and test datasets

In [ ]:
# define transforms

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



In [ ]:
# load same indices of 30% training set data taken in 01_captioning notebook

dataset_train = torchvision.datasets.OxfordIIITPet(root = os.path.join(PROJECT_ROOT, "data", "raw"),
                                             split = "trainval",
                                             transform=transform_train,
                                             download = True
                                             )


train_small_idx = np.load(
    os.path.join(PROJECT_ROOT, "data", "splits", "train_small_indices.npy")
)

dataset_train_small = Subset(dataset_train, train_small_idx)

In [ ]:
len(dataset_train)

In [ ]:
len(dataset_train_small)

In [ ]:
dataset_train_small[20]

In [ ]:
dataset_train_small[1009][0]

In [ ]:
# load test dataset
dataset_test = torchvision.datasets.OxfordIIITPet(root = os.path.join(PROJECT_ROOT, "data", "raw"),
                                             split = "test",
                                             transform=transform_test,
                                             download = True
                                             )

In [ ]:
# attach transforms
dataset_train_small.dataset.transform = transform_train
dataset_test.transform = transform_test

In [ ]:
len(dataset_test)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# define and attach transforms

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset_train_small.dataset.transform = transform_train
dataset_test.transform = transform_test


# Build Synthetic Dataset

In [ ]:
# load metadata

CHECKPOINT_FILE = os.path.join(
    PROJECT_ROOT,
    "data",
    "synthetic",
    "generation_metadata_final.json"
)

with open(CHECKPOINT_FILE, "r") as f:
    generation_metadata = json.load(f)

In [ ]:
generation_metadata

In [ ]:
len(generation_metadata)

In [ ]:
generation_metadata['0'][0]

In [ ]:
generation_metadata['0'][1]

In [ ]:
# class mapping
class_to_idx = dataset_train.class_to_idx

In [ ]:
# create Synthetic Dataset class
class SyntheticDataset(torch.utils.data.Dataset):
    def __init__(self, metadata, class_to_idx, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = class_to_idx

        for idx in metadata:
            for item in metadata[idx]:
                path = item["image_path"]
                class_name = item["class_name"]

                if class_name in class_to_idx:
                    label = class_to_idx[class_name]
                    self.samples.append((path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
# create synthetic dataset
synthetic_dataset = SyntheticDataset(
    generation_metadata,
    class_to_idx,
    transform=transform_train
)


# Create Baseline and Augmented Datasets

In [ ]:
train_baseline = dataset_train_small

In [ ]:
train_augmented = ConcatDataset([
    dataset_train_small,
    synthetic_dataset
])

In [ ]:
print("Baseline size:", len(train_baseline))
print("Synthetic size:", len(synthetic_dataset))
print("Augmented size:", len(train_augmented))

# DataLoaders

In [ ]:
BATCH_SIZE = 64

train_loader_baseline = DataLoader(
    train_baseline,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

train_loader_augmented = DataLoader(
    train_augmented,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

test_loader = DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)


# Define Model

In [ ]:
num_classes = 37

def create_model():
    model = models.resnet18(pretrained=True)
    model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


# Training Function

In [ ]:
def train_model(model, train_loader, epochs=5):

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.train()

    for epoch in range(epochs):
        running_loss = 0

        for images, labels in tqdm(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

    return model


# Evaluation Function

In [ ]:
def evaluate_model(model, loader):

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report_dict = classification_report(all_labels, all_preds, output_dict=True)

    return accuracy, report_dict


# Train Baseline

In [ ]:
model_baseline = create_model()
model_baseline = train_model(model_baseline, train_loader_baseline, epochs=5)

In [ ]:
acc_baseline, report_baseline = evaluate_model(model_baseline, test_loader)

print("Baseline Accuracy:", acc_baseline)
print(report_baseline)


# Train Augmented

In [ ]:
model_augmented = create_model()
model_augmented = train_model(model_augmented, train_loader_augmented, epochs=5)

In [ ]:
acc_augmented, report_augmented = evaluate_model(model_augmented, test_loader)

print("Augmented Accuracy:", acc_augmented)
print(report_augmented)

# Final Result

In [ ]:
# save metrics + experiment info
results = {
    "baseline": {
        "accuracy": acc_baseline,
        "report": report_baseline
    },
    "augmented": {
        "accuracy": acc_augmented,
        "report": report_augmented
    },
    "experiment_info": {
        "train_size_real": len(train_baseline),
        "train_size_synthetic": len(synthetic_dataset),
        "test_size": len(dataset_test),
        "epochs": 5,
        "batch_size": BATCH_SIZE,
        "model": "ResNet18"
    }
}


In [ ]:
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODEL_DIR, exist_ok=True)


with open(os.path.join(MODEL_DIR, "evaluation_results.json"), "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
# save models weights
torch.save(
    model_baseline.state_dict(),
    os.path.join(MODEL_DIR, "resnet18_baseline.pth")
)

torch.save(
    model_augmented.state_dict(),
    os.path.join(MODEL_DIR, "resnet18_augmented.pth")
)

In [ ]:
with open(os.path.join(MODEL_DIR, "evaluation_results.json"), "r") as f:
    results = json.load(f)

print(results["baseline"]["accuracy"])
print(results["augmented"]["accuracy"])

# Conclusion

We observed a meaningful improvement in the 37-class fine-grained breed classification task through the use of Synthetic Data Augmentation.

The baseline model was trained on 1,104 real images (30% of the available training data), while the augmented model was trained on the same 1,104 real images plus 2,181 diffusion-generated synthetic images (approximately a 2× data augmentation factor).


The augmented model demonstrated:

- Improved generalization to unseen real test data

- Higher macro-averaged F1-score, indicating more balanced performance across breeds

- Higher weighted F1-score

- An increase in overall accuracy from 82.2% to 85.6%


Specifically, the augmented model achieved an accuracy improvement of 3.4% over the baseline trained on limited real data. Additionally, macro F1-score increased by approx. 3.7%, indicating improved performance across breeds.

These results indicate that diffusion-generated synthetic images effectively enriched the training distribution and enhanced the model’s ability to generalize to unseen samples. By increasing intra-class variability and introducing additional pose, texture, and appearance diversity, the synthetic data helped the model learn more robust and discriminative feature representations.